# 00 Common Utils
This notebook contains shared functions for the KiotViet SDV pipeline.

In [ ]:
import pandas as pd
import os
import json
import logging

def setup_logging(log_path='data/warehouse/pipeline.log'):
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_path),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger('KiotVietSDV')

def detect_header_row(file_path, sheet_name=0, keyword='Mã'):
    """Detects the row index containing the header based on a keyword."""
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    for i, row in df.iterrows():
        if row.astype(str).str.contains(keyword).any():
            return i
    return 0

def load_template(file_path, keyword='Mã'):
    header_idx = detect_header_row(file_path, keyword=keyword)
    return pd.read_excel(file_path, header=header_idx)

def export_chunks(df, base_name, output_dir='data/out/excel', chunk_size=1000):
    """Exports dataframe in chunks of 1000 rows as per KiotViet import limit."""
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    num_chunks = (len(df) // chunk_size) + 1
    for i in range(num_chunks):
        chunk = df.iloc[i*chunk_size : (i+1)*chunk_size]
        if not chunk.empty:
            chunk.to_excel(f"{output_dir}/{base_name}_{i+1:03d}.xlsx", index=False)
            print(f"Exported chunk {i+1} to {output_dir}")